# Modulo 7 · Lezione 4 — Classificazione e metriche

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Regressione Logistica: come predice le categorie? |
| 2 | Matrice di confusione |
| 3 | Accuracy, Precision, Recall e F1 |
| 4 | Curva ROC e AUC |
| 5 | Classi sbilanciate |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score
)

np.random.seed(42)
print("Librerie pronte")

---
## 1. Regressione Logistica

```
Regressione lineare → predice un numero continuo (-∞, +∞)
Regressione logistica → predice una PROBABILITÀ [0, 1] usando la funzione sigmoide:

  P(y=1 | x) = 1 / (1 + e^(-z))    dove z = β₀ + β₁x₁ + ... + βₙxₙ

Classificazione: se P(y=1) ≥ soglia (default 0.5) → classe 1, altrimenti → classe 0

Perché non usare la regressione lineare per classificare?
  - Può predire valori fuori da [0,1]
  - Un campione estremo può distorcere il confine di decisione
```

In [ ]:
# Dataset: predire se un'email è spam
np.random.seed(42)
n = 1000
n_palabras_spam = np.random.exponential(15, n)
n_enlaces       = np.random.poisson(3, n)
urgencia        = np.random.uniform(0, 10, n)

z = -4 + 0.15*n_palabras_spam + 0.5*n_enlaces + 0.3*urgencia
prob_spam = 1 / (1 + np.exp(-z))
spam = (np.random.rand(n) < prob_spam).astype(int)

df = pd.DataFrame({
    'palabras_spam': n_palabras_spam,
    'n_enlaces':     n_enlaces,
    'urgencia':      urgencia,
    'spam':          spam
})
print(f"Dataset: {n} email, {spam.sum()} spam ({spam.mean()*100:.1f}%)")

X = df[['palabras_spam', 'n_enlaces', 'urgencia']]
y = df['spam']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

modelo = LogisticRegression(random_state=42)
modelo.fit(X_tr_s, y_tr)

print(f"\nCoefficienti:")
for feat, coef in zip(X.columns, modelo.coef_[0]):
    print(f"  {feat:20}: {coef:+.4f}")
print(f"Intercetta: {modelo.intercept_[0]:.4f}")

# Probabilità
probas = modelo.predict_proba(X_te_s)
print(f"\nPrime 5 predizioni:")
print(f"{'Prob No Spam':>14} {'Prob Spam':>12} {'Predizione':>12} {'Reale':>8}")
for i in range(5):
    pred = modelo.predict([X_te_s[i]])[0]
    print(f"{probas[i,0]:>14.4f} {probas[i,1]:>12.4f} {pred:>12} {y_te.iloc[i]:>8}")

# Funzione sigmoide
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
z_vals = np.linspace(-6, 6, 100)
sigmoid = 1 / (1 + np.exp(-z_vals))
axes[0].plot(z_vals, sigmoid, 'b-', lw=2)
axes[0].axhline(0.5, color='red', ls='--', lw=1, alpha=0.7, label='soglia=0.5')
axes[0].axvline(0, color='gray', ls='-', lw=0.5)
axes[0].fill_between(z_vals, 0, sigmoid, where=z_vals>0, alpha=0.1, color='green')
axes[0].set_title('Funzione sigmoide: z → P(spam)')
axes[0].set_xlabel('z = β₀ + Σβᵢxᵢ'); axes[0].set_ylabel('Probabilità')
axes[0].legend()

axes[1].scatter(X_te_s[:,0], probas[:,1], c=y_te.values, cmap='bwr', alpha=0.5, s=20)
axes[1].axhline(0.5, color='k', ls='--', lw=2, label='Soglia 0.5')
axes[1].set_xlabel('Parole spam (scalate)'); axes[1].set_ylabel('P(spam)')
axes[1].set_title('Probabilità predetta vs feature principale')
axes[1].legend()
plt.tight_layout(); plt.show()

---
## 2. Matrice di confusione

```
                   PREDIZIONE
              │  Negativo  │  Positivo  │
    ──────────┼────────────┼────────────┤
    Negativo  │     TN     │     FP     │  ← No Spam reale
    ──────────┼────────────┼────────────┤
    Positivo  │     FN     │     TP     │  ← Spam reale

TN (True Negative):  email normale, predetta normale ✓
FP (False Positive): email normale, predetta spam ✗ (falso allarme)
FN (False Negative): spam, predetta normale ✗ (spam non rilevato)
TP (True Positive):  spam, predetta spam ✓
```

In [ ]:
y_pred = modelo.predict(X_te_s)
cm = confusion_matrix(y_te, y_pred)

print("Matrice di confusione:")
print(f"                Predetto No Spam | Predetto Spam")
print(f"Reale No Spam:   TN={cm[0,0]:3d}              FP={cm[0,1]:3d}")
print(f"Reale Spam:      FN={cm[1,0]:3d}              TP={cm[1,1]:3d}")

# Visualizzazione come heatmap
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['No Spam', 'Spam'])
axes[0].set_yticklabels(['No Spam', 'Spam'])
axes[0].set_xlabel('Predizione'); axes[0].set_ylabel('Reale')
for i in range(2):
    for j in range(2):
        texto = f"{cm[i,j]}\n{'TN FP FN TP'.split()[i*2+j]}"
        axes[0].text(j, i, f"{cm[i,j]}", ha='center', va='center', fontsize=14,
                    color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Matrice di Confusione')

# Normalizzata
cm_norm = cm / cm.sum(axis=1, keepdims=True)
im2 = axes[1].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels(['No Spam', 'Spam'])
axes[1].set_yticklabels(['No Spam', 'Spam'])
axes[1].set_xlabel('Predizione'); axes[1].set_ylabel('Reale')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{cm_norm[i,j]:.2%}", ha='center', va='center', fontsize=13,
                    color='white' if cm_norm[i,j] > 0.5 else 'black')
plt.colorbar(im2, ax=axes[1])
axes[1].set_title('Matrice di Confusione (normalizzata)')
plt.tight_layout(); plt.show()

---
## 3. Metriche di classificazione

```
ACCURACY:   (TP + TN) / (TP + TN + FP + FN)
  → % di predizioni corrette
  ✗ Ingannevole con classi sbilanciate: 95% di accuracy se il 95% è classe 0

PRECISION:  TP / (TP + FP)
  → Di quelli che ho predetto positivi, quanti lo sono realmente?
  → Conta quando i FP sono costosi: sistema anti-spam → non voglio bloccare email valide

RECALL:     TP / (TP + FN)
  → Di tutti i positivi reali, quanti ne ho rilevati?
  → Conta quando i FN sono costosi: diagnosi di cancro → non voglio lasciarne senza rilevare

F1:         2 × (Precision × Recall) / (Precision + Recall)
  → Media armonica di Precision e Recall
  → Equilibrio tra le due
  → Utile con classi sbilanciate

Tradeoff Precision-Recall: alzare una abbassa l'altra (controllato dalla soglia di classificazione)
```

In [ ]:
tn, fp, fn, tp = cm.ravel()

print("=== Metriche calcolate manualmente ===")
acc   = (tp + tn) / (tp + tn + fp + fn)
prec  = tp / (tp + fp)
rec   = tp / (tp + fn)
f1    = 2 * prec * rec / (prec + rec)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}  (di quelli rilevati come spam, il {prec*100:.1f}% lo era davvero)")
print(f"Recall:    {rec:.4f}  (rilevo il {rec*100:.1f}% dello spam reale)")
print(f"F1:        {f1:.4f}")

print("\n=== Verifica con sklearn ===")
print(f"accuracy_score:  {accuracy_score(y_te, y_pred):.4f}")
print(f"precision_score: {precision_score(y_te, y_pred):.4f}")
print(f"recall_score:    {recall_score(y_te, y_pred):.4f}")
print(f"f1_score:        {f1_score(y_te, y_pred):.4f}")

print("\n=== Report completo ===")
print(classification_report(y_te, y_pred, target_names=['No Spam', 'Spam']))

# Effetto della soglia su Precision/Recall
probas_spam = modelo.predict_proba(X_te_s)[:, 1]
umbrales = np.arange(0.1, 0.95, 0.05)
precisiones, recalls, f1s = [], [], []

for u in umbrales:
    y_pred_u = (probas_spam >= u).astype(int)
    if y_pred_u.sum() == 0: continue
    precisiones.append(precision_score(y_te, y_pred_u, zero_division=0))
    recalls.append(recall_score(y_te, y_pred_u))
    f1s.append(f1_score(y_te, y_pred_u))

fig, ax = plt.subplots(figsize=(10, 4))
u_plot = umbrales[:len(precisiones)]
ax.plot(u_plot, precisiones, 'b-o', ms=4, label='Precision')
ax.plot(u_plot, recalls,     'r-o', ms=4, label='Recall')
ax.plot(u_plot, f1s,         'g-o', ms=4, label='F1')
ax.axvline(0.5, color='gray', ls='--', alpha=0.7, label='Soglia predefinita')
ax.set_xlabel('Soglia di classificazione'); ax.set_ylabel('Metrica')
ax.set_title('Tradeoff Precision-Recall in base alla soglia')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 4. Curva ROC e AUC

```
ROC (Receiver Operating Characteristic):
  Rappresenta il tradeoff tra:
    - TPR (True Positive Rate) = Recall = TP/(TP+FN)  [asse Y]
    - FPR (False Positive Rate)= FP/(FP+TN)           [asse X]
  per tutte le soglie possibili.

AUC (Area Under the Curve):
  - AUC = 1.0: modello perfetto
  - AUC = 0.5: modello casuale (linea diagonale)
  - AUC < 0.5: peggio del caso (invertire le predizioni)

Quando usare l'AUC al posto dell'Accuracy?
  Quando le classi sono sbilanciate o la soglia è regolabile.
  L'AUC misura la qualità del ranking, non della soglia.
```

In [ ]:
fpr, tpr, thresholds = roc_curve(y_te, probas_spam)
auc = roc_auc_score(y_te, probas_spam)

# Confronto con baseline casuale
np.random.seed(0)
prob_random = np.random.rand(len(y_te))
fpr_r, tpr_r, _ = roc_curve(y_te, prob_random)
auc_r = roc_auc_score(y_te, prob_random)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
axes[0].plot(fpr, tpr, 'b-', lw=2, label=f'LogisticRegression (AUC={auc:.4f})')
axes[0].plot(fpr_r, tpr_r, 'r--', lw=1, label=f'Casuale (AUC={auc_r:.4f})')
axes[0].plot([0,1],[0,1], 'k--', lw=1, alpha=0.5)
axes[0].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[0].set_xlabel('FPR (Tasso di Falsi Positivi)')
axes[0].set_ylabel('TPR (Recall)')
axes[0].set_title('Curva ROC')
axes[0].legend()

# Soglia ottimale (massimizzare TPR - FPR)
youden_idx = np.argmax(tpr - fpr)
umbral_opt = thresholds[youden_idx]
axes[0].scatter(fpr[youden_idx], tpr[youden_idx], color='red', s=100, zorder=5,
               label=f'Ottimale (soglia={umbral_opt:.3f})')
axes[0].legend(fontsize=8)

print(f"AUC: {auc:.4f}")
print(f"Soglia ottimale (Youden): {umbral_opt:.4f}")

# Confronto metriche con soglia 0.5 vs ottimale
for umbral, nombre in [(0.5, 'Soglia 0.5'), (umbral_opt, f'Soglia ottimale ({umbral_opt:.3f})')]:
    y_pred_u = (probas_spam >= umbral).astype(int)
    axes[1].bar(
        [f'{nombre}\nPrecision', f'{nombre}\nRecall', f'{nombre}\nF1'],
        [precision_score(y_te, y_pred_u), recall_score(y_te, y_pred_u), f1_score(y_te, y_pred_u)],
        alpha=0.7
    )

axes[1].set_ylim(0, 1); axes[1].set_title('Metriche per soglia')
axes[1].set_ylabel('Valore')
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

---
## 5. Classi sbilanciate

In [ ]:
# Dataset molto sbilanciato: 5% di frode
np.random.seed(42)
n_total = 1000
n_fraude = 50  # solo 5%

X_norm   = np.random.randn(n_total - n_fraude, 5)
X_fraude = np.random.randn(n_fraude, 5) + 1.5  # leggermente diverso
X_imb    = np.vstack([X_norm, X_fraude])
y_imb    = np.array([0]*(n_total-n_fraude) + [1]*n_fraude)

X_tr, X_te, y_tr, y_te = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb)

print("=== Problema con classi sbilanciate ===")

# Baseline: predice sempre la classe maggioritaria
y_baseline = np.zeros(len(y_te), dtype=int)
print(f"Baseline (predice sempre 0):")
print(f"  Accuracy:  {accuracy_score(y_te, y_baseline):.4f}  ← sembra buono!")
print(f"  F1(frode): {f1_score(y_te, y_baseline, pos_label=1):.4f}  ← disastro!")
print(f"  Recall(frode): {recall_score(y_te, y_baseline, pos_label=1, zero_division=0):.4f}")

# LogisticRegression senza bilanciamento
m_sin = LogisticRegression(random_state=42).fit(X_tr, y_tr)
y_pred_sin = m_sin.predict(X_te)

# LogisticRegression con class_weight='balanced'
m_bal = LogisticRegression(class_weight='balanced', random_state=42).fit(X_tr, y_tr)
y_pred_bal = m_bal.predict(X_te)

print(f"\nLogisticRegression SENZA bilanciamento:")
print(classification_report(y_te, y_pred_sin, target_names=['Normale','Frode']))

print(f"LogisticRegression CON class_weight='balanced':")
print(classification_report(y_te, y_pred_bal, target_names=['Normale','Frode']))

print("\nAltre strategie per dati sbilanciati:")
print("  1. class_weight='balanced' nel modello (più semplice)")
print("  2. Oversampling della classe minoritaria (SMOTE)")
print("  3. Undersampling della classe maggioritaria")
print("  4. Regolare la soglia di classificazione")
print("  5. Usare F1 o AUC come metrica, NON l'accuracy")

---
## Pratica

### Esercizio 1 — Titanic con Logistic Regression

**Consegna:** Predici la sopravvivenza sul Titanic:

```python
import seaborn as sns
df = sns.load_dataset('titanic')[['survived','pclass','sex','age','fare','embarked']].dropna()
```

1. Preprocessa: codifica sex ed embarked, scala age e fare
2. Addestra LogisticRegression
3. Stampa accuracy, precision, recall e F1
4. Visualizza la matrice di confusione
5. Quale metrica ti sembra più importante qui? Perché?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

df = sns.load_dataset('titanic')[['survived','pclass','sex','age','fare','embarked']].dropna()

# 1. Preprocessare
df['sex_num'] = (df['sex'] == 'female').astype(int)
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)
cols_escalar = ['age', 'fare']

X = df.drop(['survived','sex'], axis=1)
y = df['survived']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler()
X_tr[cols_escalar] = sc.fit_transform(X_tr[cols_escalar])
X_te[cols_escalar] = sc.transform(X_te[cols_escalar])

# 2. Addestrare
m = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr, y_tr)
y_pred = m.predict(X_te)

# 3. Metriche
print(f"Accuracy:  {accuracy_score(y_te, y_pred):.4f}")
print(f"Precision: {precision_score(y_te, y_pred):.4f}")
print(f"Recall:    {recall_score(y_te, y_pred):.4f}")
print(f"F1:        {f1_score(y_te, y_pred):.4f}")
print(classification_report(y_te, y_pred, target_names=['Non sopravvissuto','Sopravvissuto']))

# 4. Matrice di confusione
cm = confusion_matrix(y_te, y_pred)
plt.figure(figsize=(5,4))
plt.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=14)
plt.xticks([0,1],['No','Sì']); plt.yticks([0,1],['No','Sì'])
plt.xlabel('Predetto'); plt.ylabel('Reale')
plt.title('Matrice di confusione'); plt.tight_layout(); plt.show()

# 5. Riflessione
print("\nSul Titanic, se il modello predice 'sopravvissuto' ma la persona non è sopravvissuta,")
print("la conseguenza è minore rispetto a predire 'non sopravvissuto' quando invece è sopravvissuta")
print("→ Il Recall della classe 'Sopravvissuto' è la metrica più importante qui.")
```
</details>

---
### Esercizio 2 — Effetto della soglia di classificazione

**Consegna:** Sistema di rilevamento di tumori maligni:

```python
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
```

1. Addestra LogisticRegression e ottieni le probabilità della classe positiva (maligno)
2. Prova le soglie: [0.3, 0.4, 0.5, 0.6, 0.7]
3. Per ogni soglia: Precision, Recall, F1
4. Quale soglia sceglieresti se il costo di un FN (cancro non rilevato) è molto alto?
5. Traccia la curva ROC e calcola l'AUC

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_curve, roc_auc_score)
import matplotlib.pyplot as plt
import numpy as np

data = load_breast_cancer()
X, y = data.data, data.target  # 1=benigno, 0=maligno
# Invertire: positivo = maligno (0)
y_inv = 1 - y

X_tr, X_te, y_tr, y_te = train_test_split(X, y_inv, test_size=0.2, random_state=42, stratify=y_inv)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

m = LogisticRegression(max_iter=10000).fit(X_tr_s, y_tr)
probas = m.predict_proba(X_te_s)[:, 1]  # P(maligno)

print(f"{'Soglia':>8} {'Precision':>10} {'Recall':>10} {'F1':>10}")
for u in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred = (probas >= u).astype(int)
    p = precision_score(y_te, y_pred, zero_division=0)
    r = recall_score(y_te, y_pred)
    f = f1_score(y_te, y_pred)
    print(f"{u:>8.1f} {p:>10.4f} {r:>10.4f} {f:>10.4f}")

# 4.
print("\nSe il FN (cancro non rilevato) è molto costoso → massimizzare il RECALL.")
print("→ Usare una soglia bassa (0.3): rileviamo più tumori anche se aumentano i FP (biopsie inutili).")

# 5. ROC
fpr, tpr, _ = roc_curve(y_te, probas)
auc = roc_auc_score(y_te, probas)
plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {auc:.4f}')
plt.plot([0,1],[0,1],'k--', alpha=0.5)
plt.xlabel('FPR'); plt.ylabel('TPR (Recall)')
plt.title('Curva ROC — Rilevamento tumore maligno')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()
```
</details>

---
### Esercizio 3 — Classificazione multiclasse

**Consegna:** LogisticRegression funziona anche per classi multiple (Iris ne ha 3):

1. Carica Iris e addestra LogisticRegression con `multi_class='auto'`
2. Stampa il classification_report per le 3 classi
3. Visualizza la matrice di confusione (3×3)
4. Quale classe confonde di più il modello? Perché ha senso dal punto di vista biologico?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

iris = datasets.load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

m = LogisticRegression(max_iter=1000, random_state=42)
m.fit(X_tr_s, y_tr)
y_pred = m.predict(X_te_s)

print(classification_report(y_te, y_pred, target_names=iris.target_names))

cm = confusion_matrix(y_te, y_pred)
plt.figure(figsize=(6,5))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(3), iris.target_names, rotation=30)
plt.yticks(range(3), iris.target_names)
plt.xlabel('Predetto'); plt.ylabel('Reale')
for i in range(3):
    for j in range(3):
        color = 'white' if cm[i,j] > cm.max()/2 else 'black'
        plt.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=13, color=color)
plt.colorbar(); plt.title('Matrice di confusione Iris'); plt.tight_layout(); plt.show()

print("\nVersicolor e Virginica si confondono di più tra loro:")
print("Sono le due specie più simili dal punto di vista morfologico.")
print("Setosa è perfettamente separabile (specie diversa senza sovrapposizione).")
```
</details>

---
### Esercizio 4 — Calcolare le metriche da zero

**Consegna:** Senza usare sklearn.metrics, implementa le quattro metriche:

```python
y_real = np.array([1,0,1,1,0,1,0,0,1,1,0,1,0,0,1])
y_pred = np.array([1,0,1,0,0,1,1,0,1,1,0,0,0,1,1])
```

1. Calcola TP, TN, FP, FN
2. Calcola Accuracy, Precision, Recall, F1 manualmente
3. Verifica con sklearn
4. Quale delle 4 metriche ti dà più informazioni utili in questo caso?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_real = np.array([1,0,1,1,0,1,0,0,1,1,0,1,0,0,1])
y_pred = np.array([1,0,1,0,0,1,1,0,1,1,0,0,0,1,1])

# 1. Componenti della matrice di confusione
TP = np.sum((y_pred == 1) & (y_real == 1))
TN = np.sum((y_pred == 0) & (y_real == 0))
FP = np.sum((y_pred == 1) & (y_real == 0))
FN = np.sum((y_pred == 0) & (y_real == 1))
print(f"TP={TP}, TN={TN}, FP={FP}, FN={FN}")

# 2. Metriche
acc   = (TP + TN) / len(y_real)
prec  = TP / (TP + FP)
rec   = TP / (TP + FN)
f1    = 2 * prec * rec / (prec + rec)
print(f"\nAccuracy (manuale):  {acc:.4f}")
print(f"Precision (manuale): {prec:.4f}")
print(f"Recall (manuale):    {rec:.4f}")
print(f"F1 (manuale):        {f1:.4f}")

# 3. Verificare
print(f"\nAccuracy (sklearn):  {accuracy_score(y_real, y_pred):.4f}")
print(f"Precision (sklearn): {precision_score(y_real, y_pred):.4f}")
print(f"Recall (sklearn):    {recall_score(y_real, y_pred):.4f}")
print(f"F1 (sklearn):        {f1_score(y_real, y_pred):.4f}")

# 4. Interpretazione
print("\nInterpretazione:")
print(f"  Su 15 esempi, 8 sono positivi e 7 negativi → classi quasi bilanciate.")
print(f"  Accuracy={acc:.0%}: il modello indovina il {acc:.0%} dei casi.")
print(f"  F1={f1:.4f}: equilibrio tra precision e recall.")
print("  Qui F1 è la metrica più informativa per il suo equilibrio.")
```
</details>

---
### Esercizio 5 (sfida) — Rilevatore di frode con soglia dinamica

**Consegna:** Hai un sistema di rilevamento frodi. Il costo di un FN (frode non rilevata) è 500€ e il costo di un FP (transazione legittima bloccata) è 5€.

1. Genera un dataset di frode (2% frode): `np.random.seed(5); X = np.random.randn(2000,10); y = np.random.choice([0,1], 2000, p=[0.98, 0.02])`
2. Addestra LogisticRegression con `class_weight='balanced'`
3. Prova soglie da 0.1 a 0.9 e calcola il **costo totale** per ognuna: `costo = FN*500 + FP*5`
4. Visualizza costo vs soglia
5. Qual è la soglia che minimizza il costo? Coincide con la soglia di massimo F1?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler

np.random.seed(5)
X = np.random.randn(2000, 10)
y = np.random.choice([0,1], 2000, p=[0.98, 0.02])
print(f"Frodi: {y.sum()} ({y.mean()*100:.1f}%)")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

m = LogisticRegression(class_weight='balanced', max_iter=1000).fit(X_tr_s, y_tr)
probas = m.predict_proba(X_te_s)[:,1]

umbrales = np.arange(0.05, 0.96, 0.05)
costes, f1s = [], []
for u in umbrales:
    y_pred = (probas >= u).astype(int)
    if y_pred.sum() == 0: continue
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    costes.append(fn*500 + fp*5)
    f1s.append(f1_score(y_te, y_pred))

u_plot = umbrales[:len(costes)]
u_min_coste = u_plot[np.argmin(costes)]
u_max_f1    = u_plot[np.argmax(f1s)]

fig, axes = plt.subplots(1, 2, figsize=(14,4))
axes[0].plot(u_plot, costes, 'r-o', ms=4)
axes[0].axvline(u_min_coste, color='blue', ls='--', label=f'Ottimale costo={u_min_coste:.2f}')
axes[0].set_title('Costo economico per soglia'); axes[0].set_xlabel('Soglia')
axes[0].set_ylabel('Costo (€)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(u_plot, f1s, 'g-o', ms=4)
axes[1].axvline(u_max_f1, color='blue', ls='--', label=f'Ottimale F1={u_max_f1:.2f}')
axes[1].set_title('F1 per soglia'); axes[1].set_xlabel('Soglia')
axes[1].set_ylabel('F1'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Soglia costo minimo: {u_min_coste:.2f} (costo={min(costes):,.0f}€)")
print(f"Soglia F1 massimo:    {u_max_f1:.2f}")
print("\nNon coincidono necessariamente — F1 ignora i costi asimmetrici.")
print("In produzione: ottimizzare la metrica di business, non la metrica statistica.")
```
</details>

---
## Riepilogo

```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

m = LogisticRegression(class_weight='balanced').fit(X_tr, y_tr)
y_pred = m.predict(X_te)
probas = m.predict_proba(X_te)[:, 1]  # probabilità classe positiva

# Metriche
print(classification_report(y_te, y_pred))
auc = roc_auc_score(y_te, probas)

# Soglia personalizzata
y_pred_custom = (probas >= 0.3).astype(int)

# Scegliere la metrica in base al contesto:
# Classi bilanciate:      Accuracy o F1
# Classi sbilanciate:     F1, AUC-ROC
# FN molto costoso:       Recall
# FP molto costoso:       Precision
```

## Cosa vediamo nella prossima lezione?
Nella **Lezione 5** impariamo gli **Alberi di Decisione** e **Random Forest**, modelli molto più potenti della regressione logistica per la maggior parte dei problemi.